In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from utils import to_snake_case

In [ ]:
to_snake_case(['testIdVar'])

In [ ]:
titanic = pd.read_csv('../datasets/titanic.csv')

In [ ]:
column_names = to_snake_case(titanic.columns.values.tolist())
column_names

In [ ]:
titanic.rename(columns=column_names, inplace=True)

In [ ]:
titanic.info()

In [ ]:
titanic.head()

#### 1. What's the overall survival rate? (mean() of Survived)

In [ ]:
titanic['survived'].mean() * 100

In [ ]:
titanic.describe(include=np.number)['survived']['mean'] * 100

#### 2. What's the survival rate by pclass?

In [ ]:
survival_rate_by_pclass = (
    titanic.
        groupby(['pclass'])
        .agg({'survived': ['mean', 'sum', 'count']})
        .reset_index()
        .droplevel(level=0, axis=1)
)

In [ ]:
columns = ['passenger_class', 'survival_rate', 'survivors', 'total_passengers']
survival_rate_by_pclass.columns = columns

In [ ]:
survival_rate_by_pclass['survival_rate'] = survival_rate_by_pclass['survival_rate'].apply(lambda x: f'{x*100:0.2f} %')

In [ ]:
survival_rate_by_pclass

#### 3. Find average age, fare, and number of siblings/spouses

In [ ]:
titanic[['age', 'fare', 'sib_sp']].mean()

#### 4. Count missing values in each column

In [ ]:
nulls_count = titanic.isna().sum()
missing_values = nulls_count[nulls_count > 0]
missing_values

In [ ]:
total_counts = titanic[nulls_count[nulls_count > 0].index].count()
total_counts

In [ ]:
missing_df = pd.DataFrame(
    {
        'missing_value_count': titanic.isnull().sum(),
        'total_count': titanic.count() ,
        'percentage': titanic.isnull().sum() * 100 / titanic.count()
    }
)

missing_df = missing_df[missing_df['missing_value_count'] > 0]
missing_df['percentage'] = missing_df['percentage'].apply(lambda x: f'{x:.2f}%')
missing_df

#### 5. Show unique values in Pclass, Sex, Embarked

In [ ]:
titanic.loc[:,['pclass', 'sex', 'embarked']].nunique()

In [ ]:
distinct_combination = titanic.groupby(['pclass', 'sex', 'embarked']).size().reset_index(name='count')
distinct_combination

In [ ]:
distinct_combination.shape[0]

In [ ]:
titanic[['pclass', 'sex', 'embarked']].drop_duplicates()

In [ ]:
columns=['pclass', 'sex', 'embarked']
for col in columns:
    u = titanic[col].drop_duplicates().tolist()
    print(f'Values of column [{col}]: {u}')

#### 6. Select all female passengers who survived

In [ ]:
titanic[(titanic['sex']=='female') & (titanic['survived']==1)].head()

In [ ]:
titanic.query("sex=='female' and survived==1").head()

#### 7. Show passengers who paid > $50 fare and under 18 years old and first-class passengers with no siblings/spouses

In [ ]:
mask = (titanic['fare']>50) & (titanic['age']<18) & (titanic['pclass']==1) & (titanic['sib_sp']==0)
titanic.loc[mask]

#### 8. Calculate survival rate by gender

In [ ]:
titanic.groupby(['sex']).agg({'survived': ['mean', 'sum', 'count']}).droplevel(axis=1, level=0).reset_index()

#### 9. Count passengers by embarkation port (Embarked)

In [ ]:
titanic.groupby(['embarked'], dropna=False).agg({'passenger_id': ['count', 'nunique']}).reset_index()

#### 10. Create age groups [(<=12, child), (12-18, Teen), (18-30, Young Adult), (30-50, Adult), (>50, Senior)]

In [ ]:
# age groups
bins = [0, 12, 18, 30, 50, 200]
labels = ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
titanic['age_group'] = pd.cut(titanic['age'], bins=bins, labels=labels, right=True)
titanic[['age', 'age_group']].head()

In [ ]:
conditions = [
    titanic['age'] <= 12,
    titanic['age'].between(12, 18, inclusive='right'),
    titanic['age'].between(18, 30, inclusive='right'),
    titanic['age'].between(30, 50, inclusive='right'),
    titanic['age'] > 50
]

titanic['age_group'] = np.select(conditions, labels, default='Unknown')
titanic[['age', 'age_group']].head()

#### 11. Which age group had the most survivors?

In [ ]:
(titanic.loc[titanic['survived']==1]
    .groupby(['age_group'])
    .agg({'age_group': 'count'})
    .rename(columns={'age_group': 'count'})
    .reset_index()
)

In [ ]:
titanic['total_count'] = titanic.groupby(['age_group'])['age_group'].transform('count')

(titanic.loc[titanic['survived']==1]
    .groupby(['age_group'])
    .agg(
        survived_count = ('age_group', 'count'),
        total_passengers=('total_count', 'first')
    )
    .reset_index()
)

#### 12. Find number of passengers sharing same ticket

In [ ]:
titanic['ticket_count'] = titanic.groupby(['ticket'])['ticket'].transform('count')
titanic[titanic['ticket_count']>1].shape[0]

In [ ]:
ticket_count = titanic['ticket'].value_counts().reset_index()
ticket_count[ticket_count['count']>1]['count'].sum()

#### 13. Find passenger ids of people who are sharing same ticket

In [ ]:
ticket_count = titanic['ticket'].value_counts().reset_index()
more_than_one_ticket = ticket_count[ticket_count['count']>1]
(titanic
    .merge(more_than_one_ticket, how='inner', left_on='ticket', right_on='ticket')[['passenger_id', 'ticket']]
    .sort_values(by='ticket', ascending=True)
)

#### 14. Fill (Impute) missing Age with median age by Pclass

In [ ]:
titanic['age'].isna().sum()

In [ ]:
titanic['age_'] = titanic['age'].fillna(titanic['age'].mean().round(0))

In [ ]:
titanic['age_'].isna().sum()

#### 15. Create Title from Name (Mr, Mrs, Miss, Master, etc.)

In [ ]:
titanic['name'].str.extract(r' ([A-Za-z]+)\.', expand=False).value_counts()

#### 16. Create Is_Child (age < 18)

In [ ]:
titanic['is_child'] = np.where(titanic['age']<18, 1, 0)
titanic[['age', 'is_child']].loc[titanic['age'].between(10, 22)]